<a href="https://colab.research.google.com/github/yutaota/intro_to_causal_inference/blob/main/Ex04_G_Computation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. データ生成

In [14]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor

# 共変量の生成
np.random.seed(1)
N = 1000
X = np.random.randn(N, 5)
X[:, 0] = np.random.exponential(1, N) - 1

# 処置割当：X1, X2, X3 に依存
prob_t = 1 / (1 + np.exp(-(0.5 * X[:, 0] + 0.8 * X[:, 1] + 0.3 * X[:, 2])))
A = np.random.binomial(1, prob_t)

# アウトカム：X1, X2, X3, Uに依存
beta_0 = 2
delta = 1.5
U = 0.5 * X[:, 3] - 0.2 * X[:, 4] + np.random.rand(N)
Y = beta_0 * A + delta * A * X[:, 0]**2 + 1.2 * X[:, 0] + 0.6 * X[:, 1] - 0.3 * X[:, 2] + U

true_ATE_population = beta_0 + delta * 1.0

print(f'サンプル数: {N}（処置群: {A.sum()}、非処置群: {N - A.sum()}）')
print(f'真の母集団ATE: {true_ATE_population:.3f}')

data = pd.DataFrame(X[:, 0:3], columns=['X1', 'X2', 'X3'])
data['A'] = A
data['Y'] = Y


サンプル数: 1000（処置群: 509、非処置群: 491）
真の母集団ATE: 3.500


### 2. g-computationによるATE推定


- **重回帰**：各群で $Y$ を $X_1,X_2,X_3$ に回帰
- **ランダムフォレスト回帰**：各群で柔軟な予測モデルを学習

In [15]:
def estimate_gcomp(df, method):

    Xcols = ['X1', 'X2', 'X3']
    RF_TREES = 200


    """ Step 1. 群別モデルを学習 """
    df1 = df.loc[df['A'] == 1]
    df0 = df.loc[df['A'] == 0]

    if method == 'linear':
        mod1 = sm.OLS(df1['Y'], sm.add_constant(df1[Xcols], has_constant='add')).fit()
        mod0 = sm.OLS(df0['Y'], sm.add_constant(df0[Xcols], has_constant='add')).fit()
        X_all = sm.add_constant(df[Xcols], has_constant='add')
    elif method == 'rf':
        mod1 = RandomForestRegressor(n_jobs=4).fit(df1[Xcols], df1['Y'])
        mod0 = RandomForestRegressor(n_jobs=4).fit(df0[Xcols], df0['Y'])
        X_all = df[Xcols]
    else:
        raise ValueError("method は 'linear' または 'rf' を指定してください。")

    """ Step 2. 予測値の取得 """
    y1_hat = mod1.predict(X_all)
    y0_hat = mod0.predict(X_all)

    """ Step 3. 平均してATEを推定 """
    ate = float(np.mean(y1_hat - y0_hat))

    return ate


ate_linear = estimate_gcomp(data, 'linear')
ate_rf = estimate_gcomp(data, 'rf')

print(f'g-computation（線形回帰）: {ate_linear:.3f}')
print(f'g-computation（ランダムフォレスト回帰）: {ate_rf:.3f}')

g-computation（線形回帰）: 2.985
g-computation（ランダムフォレスト回帰）: 3.438


### 3. ブートストラップ95%信頼区間

In [13]:
from tqdm import tqdm

B = 100
rng = np.random.default_rng(1)  # ブートストラップ用の独立した乱数

boot_linear = np.empty(B)
boot_rf = np.empty(B)

for b in tqdm(range(B)):
    indices = rng.integers(0, len(data), size=len(data))
    bootstrap_sample = data.iloc[indices].reset_index(drop=True)
    boot_linear[b] = estimate_gcomp(bootstrap_sample, 'linear')
    boot_rf[b] = estimate_gcomp(bootstrap_sample, 'rf')

ci_linear = np.quantile(boot_linear, [0.025, 0.975])
ci_rf = np.quantile(boot_rf, [0.025, 0.975])

print(f'真の母集団ATE: {true_ATE_population:.3f}')
print(f'線形回帰:        ATE = {ate_linear:.3f}, 95% CI = [{ci_linear[0]:.3f}, {ci_linear[1]:.3f}]')
print(f'ランダムフォレスト回帰: ATE = {ate_rf:.3f}, 95% CI = [{ci_rf[0]:.3f}, {ci_rf[1]:.3f}]')

# result = pd.DataFrame({
#     '推定方法': ['線形回帰', 'ランダムフォレスト'],
#     'ATE推定値': [ate_linear, ate_rf],
#     '95% CI 下限': [ci_linear[0], ci_rf[0]],
#     '95% CI 上限': [ci_linear[1], ci_rf[1]],
# }).round(3)

# print(result.to_string(index=False))

100%|██████████| 40/40 [00:28<00:00,  1.40it/s]

真の母集団ATE: 3.500
群別線形回帰:        ATE = 2.985, 95% CI = [2.836, 3.242]
群別ランダムフォレスト: ATE = 3.438, 95% CI = [3.217, 3.775]
